<a href="https://colab.research.google.com/github/guilhermemoraes-lasalle/ColecoesEassociacoes/blob/main/Atividade_Visualizacao_Geoespacial_Folium_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade Prática — Visualização Geoespacial com Folium

## Inteligência Geográfica — Oportunidades Imobiliárias

Neste notebook será criada uma análise geoespacial de imóveis localizados nos municípios de **Nova Iguaçu** e **Queimados**, utilizando Python, Pandas e Folium.

### Objetivos
- Criar mapas interativos;
- centralizar mapas a partir da média das coordenadas;
- adicionar marcadores com informações dos imóveis;
- utilizar `CircleMarker`;
- diferenciar imóveis por cidade;
- agrupar pontos com `MarkerCluster`;
- personalizar ícones de acordo com o tipo do imóvel;
- exportar o mapa final para HTML.


## 1. Instalação e Importação das Bibliotecas


In [1]:
!pip -q install folium


In [2]:
import pandas as pd
import numpy as np
import folium

from folium.plugins import MarkerCluster

print("Bibliotecas carregadas com sucesso!")


Bibliotecas carregadas com sucesso!


## 2. Criação da Base de Dados


In [3]:
# Gerando dados sintéticos de imóveis em Nova Iguaçu e Queimados
np.random.seed(42)
n_imoveis = 45

# Coordenadas base aproximadas:
# Nova Iguaçu: -22.756, -43.460
# Queimados: -22.716, -43.555

dados_imoveis = {
    'id_imovel': range(1, n_imoveis + 1),
    'cidade': np.where(
        np.random.rand(n_imoveis) > 0.4,
        'Nova Iguaçu',
        'Queimados'
    ),
    'valor_venda': np.random.uniform(
        150000,
        850000,
        n_imoveis
    ).round(2),
    'tipo': np.random.choice(
        ['Casa', 'Apartamento', 'Terreno'],
        n_imoveis
    )
}

df_mapa = pd.DataFrame(dados_imoveis)


# Função para gerar latitude
def gerar_lat(cidade):
    if cidade == 'Nova Iguaçu':
        return -22.756 + np.random.uniform(-0.03, 0.03)

    return -22.716 + np.random.uniform(-0.02, 0.02)


# Função para gerar longitude
def gerar_lon(cidade):
    if cidade == 'Nova Iguaçu':
        return -43.460 + np.random.uniform(-0.03, 0.03)

    return -43.555 + np.random.uniform(-0.02, 0.02)


df_mapa['latitude'] = df_mapa['cidade'].apply(gerar_lat)
df_mapa['longitude'] = df_mapa['cidade'].apply(gerar_lon)

print("Base de imóveis criada com sucesso!")

display(df_mapa.head(10))


Base de imóveis criada com sucesso!


,id_imovel,cidade,valor_venda,tipo,latitude,longitude
0,1,Queimados,613765.60,Casa,-22.714426,-43.571388
1,2,Nova Iguaçu,368197.75,Terreno,-22.737554,-43.439882
2,3,Nova Iguaçu,514047.61,Apartamento,-22.732235,-43.470753
3,4,Nova Iguaçu,532697.20,Terreno,-22.766920,-43.478809
4,5,Queimados,279398.12,Casa,-22.731598,-43.573369
5,6,Queimados,828709.24,Casa,-22.726883,-43.551364
6,7,Queimados,692592.98,Apartamento,-22.718916,-43.547897
7,8,Nova Iguaçu,807649.26,Terreno,-22.736919,-43.489005
8,9,Nova Iguaçu,776379.15,Terreno,-22.734356,-43.459274
9,10,Nova Iguaçu,568529.99,Apartamento,-22.785583,-43.476410


## Informações Gerais da Base


In [4]:
print("Quantidade de imóveis:", len(df_mapa))

print("\nImóveis por cidade:")
print(df_mapa['cidade'].value_counts())

print("\nImóveis por tipo:")
print(df_mapa['tipo'].value_counts())

print("\nValor médio dos imóveis por cidade:")
display(
    df_mapa.groupby('cidade')['valor_venda']
    .mean()
    .round(2)
)


Quantidade de imóveis: 45

Imóveis por cidade:
cidade
Nova Iguaçu    23
Queimados      22
Name: count, dtype: int64

Imóveis por tipo:
tipo
Terreno        18
Casa           14
Apartamento    13
Name: count, dtype: int64

Valor médio dos imóveis por cidade:


,valor_venda
cidade,
Nova Iguaçu,529256.90
Queimados,467647.23


# Parte 1 — Inicialização e Marcadores Básicos


## 3. Cálculo da Coordenada Central


In [5]:
latitude_central = df_mapa['latitude'].mean()
longitude_central = df_mapa['longitude'].mean()

print("Latitude central:", latitude_central)
print("Longitude central:", longitude_central)


Latitude central: -22.736785025907697
Longitude central: -43.50738358758272


## 4. Criação do Primeiro Mapa


In [6]:
mapa_basico = folium.Map(
    location=[
        latitude_central,
        longitude_central
    ],
    zoom_start=12,
    tiles='OpenStreetMap'
)

mapa_basico


## 5. Marcadores para os 5 Primeiros Imóveis


In [7]:
for _, imovel in df_mapa.head(5).iterrows():

    popup_texto = (
        f"<b>Tipo:</b> {imovel['tipo']}<br>"
        f"<b>Valor:</b> R$ {imovel['valor_venda']:,.2f}"
    )

    folium.Marker(
        location=[
            imovel['latitude'],
            imovel['longitude']
        ],
        popup=folium.Popup(
            popup_texto,
            max_width=250
        )
    ).add_to(mapa_basico)

mapa_basico


# Parte 2 — Customização Visual com Marcadores Circulares


## 6. Novo Mapa com `CircleMarker`


In [8]:
mapa_circular = folium.Map(
    location=[
        latitude_central,
        longitude_central
    ],
    zoom_start=12,
    tiles='OpenStreetMap'
)

for _, imovel in df_mapa.iterrows():

    # Regra de cor por cidade
    if imovel['cidade'] == 'Nova Iguaçu':
        cor = 'blue'
    else:
        cor = 'orange'

    popup_texto = (
        f"<b>ID:</b> {imovel['id_imovel']}<br>"
        f"<b>Cidade:</b> {imovel['cidade']}<br>"
        f"<b>Tipo:</b> {imovel['tipo']}<br>"
        f"<b>Valor:</b> R$ {imovel['valor_venda']:,.2f}"
    )

    folium.CircleMarker(
        location=[
            imovel['latitude'],
            imovel['longitude']
        ],
        radius=8,
        color=cor,
        fill=True,
        fill_color=cor,
        fill_opacity=0.7,
        tooltip='Clique para detalhes',
        popup=folium.Popup(
            popup_texto,
            max_width=300
        )
    ).add_to(mapa_circular)

mapa_circular


### Regras visuais utilizadas

- **Nova Iguaçu:** azul;
- **Queimados:** laranja;
- **Raio:** 8 pixels;
- **Tooltip:** "Clique para detalhes".

Ao clicar em cada ponto, são exibidos ID, cidade, tipo e valor do imóvel.


# Parte 3 — Agrupamento Inteligente com MarkerCluster


## 7. Criação do Mapa com Cluster


In [9]:
mapa_cluster = folium.Map(
    location=[
        latitude_central,
        longitude_central
    ],
    zoom_start=12,
    tiles='OpenStreetMap'
)

# Criando o agrupador
cluster = MarkerCluster(
    name='Imóveis'
)

cluster.add_to(mapa_cluster)

print("Mapa e MarkerCluster criados com sucesso!")


Mapa e MarkerCluster criados com sucesso!


## 8. Adição dos Imóveis ao Cluster


In [10]:
# Cores dos ícones por tipo de imóvel
cores_tipo = {
    'Casa': 'green',
    'Apartamento': 'blue',
    'Terreno': 'gray'
}

for _, imovel in df_mapa.iterrows():

    cor_icone = cores_tipo.get(
        imovel['tipo'],
        'red'
    )

    popup_texto = (
        f"<b>Imóvel #{imovel['id_imovel']}</b><br>"
        f"<b>Cidade:</b> {imovel['cidade']}<br>"
        f"<b>Tipo:</b> {imovel['tipo']}<br>"
        f"<b>Valor:</b> R$ {imovel['valor_venda']:,.2f}"
    )

    folium.Marker(
        location=[
            imovel['latitude'],
            imovel['longitude']
        ],
        popup=folium.Popup(
            popup_texto,
            max_width=300
        ),
        tooltip='Clique para detalhes',
        icon=folium.Icon(
            color=cor_icone,
            icon='home',
            prefix='fa'
        )
    ).add_to(cluster)

mapa_cluster


### Cores utilizadas no mapa com cluster

- 🟢 **Casa:** verde;
- 🔵 **Apartamento:** azul;
- ⚪ **Terreno:** cinza.

O `MarkerCluster` agrupa automaticamente imóveis próximos. Ao aumentar o zoom, os grupos são separados até que os marcadores individuais sejam exibidos.


## 9. Salvando o Mapa Final em HTML


In [11]:
nome_arquivo = 'mapa_imoveis_baixada.html'

mapa_cluster.save(nome_arquivo)

print(
    f"Mapa salvo com sucesso como: {nome_arquivo}"
)


Mapa salvo com sucesso como: mapa_imoveis_baixada.html


## 10. Download do Mapa no Google Colab


In [12]:
from google.colab import files

files.download('mapa_imoveis_baixada.html')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Conclusão

A atividade demonstrou como utilizar a biblioteca **Folium** para representar dados imobiliários geograficamente.

Foram construídos três mapas:

1. **Mapa básico:** cinco primeiros imóveis com marcadores e popups;
2. **Mapa com CircleMarker:** todos os imóveis, diferenciados pela cidade;
3. **Mapa com MarkerCluster:** todos os imóveis agrupados de forma inteligente e diferenciados pelo tipo.

O mapa final é salvo no arquivo:

`mapa_imoveis_baixada.html`

Como o arquivo é HTML, ele mantém os recursos interativos do Folium e pode ser aberto diretamente em um navegador.
